# pi0 — delta-action benchmark run (A100-80)

Replaces `train_pi0_v2_benchmark_runpod.ipynb`. Everything the 30k absolute-action
run got wrong or left on the table, in one place.

| | old run | this notebook |
|---|---|---|
| action space | absolute joint targets | **delta from state** (+ gripper absolute) — openpi's `make_bool_mask(6,-1)` |
| chunk sampling | every frame (0.06°/step) | **`ACTION_STRIDE`** frames (0.27° at 5; net gain w/ delta ~6.5×, measured) |
| LoRA targets | Gemma names only → SigLIP got q/k/v | **both towers** (`out_proj`, `fc1`, `fc2` added) |
| `torch.compile` | never enabled | **on** (`modeling_pi0.py:590` really calls it) |
| DataLoader | `num_workers=2`, no prefetch | **tuned + measured** before you commit |
| LR schedule | none in `common/train.py` | **warmup + cosine**, sqrt-scaled to batch |
| warm-start | n/a | **`--init-from`, stats buffers excluded** (the silent killer) |

Read `delta_joint/README.md` for *why*. The logic lives in `delta_joint/` and is
covered by 18 assertions — this notebook only wires it up, so it cannot drift from
the tested code the way the v2/v3 notebooks did.

## 0 · Pod setup

In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/SreevaatsavB/fairino-fr5-policies.git"
REPO_DIR = Path("/workspace/fairino-fr5-act-pipeline")
BRANCH   = "main"


def _git(*args, check=True):
    r = subprocess.run(["git", "-C", str(REPO_DIR), *args],
                       capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()


if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"clone failed:\n{r.stderr}")
    print(f"cloned {REPO_URL}")
else:
    # Hard-sync to origin/BRANCH. A plain `pull --ff-only` refuses whenever the
    # tree is dirty (the instruction tool's .bak files, an edited notebook, stray
    # __pycache__) and the old cell swallowed that with check=False — so the
    # notebook LOOKED updated while running week-old code, which cost a pod run
    # with "cannot import name shm_size_mb from speedups".
    _git("fetch", "origin", BRANCH)
    local, remote = _git("rev-parse", "HEAD"), _git("rev-parse", f"origin/{BRANCH}")
    if local != remote:
        dirty = _git("status", "--porcelain")
        if dirty:
            print("discarding local changes in the pod checkout:")
            for line in dirty.splitlines()[:10]:
                print(f"    {line}")
            _git("reset", "--hard", "HEAD")
            _git("clean", "-fd")
        _git("checkout", BRANCH)
        _git("reset", "--hard", f"origin/{BRANCH}")
        print(f"synced {local[:8]} -> {remote[:8]}")
    else:
        print(f"already at origin/{BRANCH}")

sys.path[:0] = [str(REPO_DIR / "common"), str(REPO_DIR / "delta_joint")]
os.chdir(REPO_DIR)

# A git sync does not touch modules this kernel already imported. DROP them from
# sys.modules rather than importlib.reload-ing: dataset_delta does
# `from dataset import FR5Dataset` at import time, so reloading dataset_delta
# BEFORE dataset re-binds the OLD class and the new argument vanishes
#     TypeError: FR5Dataset.__init__() got an unexpected keyword argument 'aug_prob'
# Popping is order-independent — the next `from X import Y` re-reads from disk.
importlib.invalidate_caches()
for _m in ("dataset", "dataset_delta", "speedups", "lerobot_patches",
           "proprio", "vla_pretrained", "rebalance_instructions",
           "action_record", "plot_actions", "convert_episodes"):
    if sys.modules.pop(_m, None) is not None:
        print(f"  dropped stale {_m}")

# Prove the working tree really is what this notebook expects. Tripping this means
# the sync above did not take — stop here rather than 40 minutes into training.
import speedups as _sp
for _need in ("shm_size_mb", "use_file_system_sharing", "loader_kwargs",
              "finetune_flags", "warmup_cosine", "init_from", "scaled_lr"):
    assert hasattr(_sp, _need), (
        f"speedups.{_need} missing -> the pod checkout is stale. "
        f"Restart the kernel and re-run this cell.")
import dataset_delta as _dd
for _need in ("DeltaJointDataset", "for_inference", "parse_action_space"):
    assert hasattr(_dd, _need), f"dataset_delta.{_need} missing -> stale checkout"

# Signature check, not just symbol presence: a stale `dataset` module still exports
# FR5Dataset, so hasattr passes while the constructor rejects the new arguments.
import inspect
import dataset as _dsmod
_params = inspect.signature(_dsmod.FR5Dataset.__init__).parameters
for _need in ("pad_resize", "aug_prob", "frame_stride"):
    assert _need in _params, (
        f"FR5Dataset.__init__ has no {_need!r} -> a stale `dataset` module is "
        f"still loaded. Restart the kernel and re-run this cell.")
# and that DeltaJointDataset really subclasses the freshly-loaded FR5Dataset
assert issubclass(_dd.DeltaJointDataset, _dsmod.FR5Dataset), (
    "DeltaJointDataset is bound to a DIFFERENT FR5Dataset than the one just "
    "loaded -> module reload ordering problem. Restart the kernel.")

# The Sanding path needs convert_episodes' max_sync_gap (blocking-gripper stall
# filter) and its dense episode_index. BRANCH above is "main", so until those
# land there this cell would happily sync a checkout that silently converts the
# data the old way: 20% frozen frames and episodes dropped from both splits.
# Fail here instead — set BRANCH to the branch carrying them, or merge it.
import convert_episodes as _ce
assert "max_sync_gap" in inspect.signature(_ce.convert).parameters, (
    f"convert_episodes.convert has no 'max_sync_gap' -> origin/{BRANCH} predates "
    f"the Sanding fixes. Point BRANCH at the branch that has them (or merge it) "
    f"and re-run this cell.")

print(f"\nHEAD  {_git('log', '--oneline', '-1')}")
print(f"clean {'yes' if not _git('status', '--porcelain') else 'NO'}")


In [ ]:
!pip -q install "lerobot==0.5.1" peft bitsandbytes wandb hf_transfer

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  {p.total_memory/1e9:.0f} GB  sm_{p.major}{p.minor}")

In [ ]:
import getpass
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"        # 10 GB pushes take minutes, not hours
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login, whoami
login(HF_TOKEN)
HF_USER = whoami()["name"]
print("logged in as", HF_USER)

## 1 · Parameters

Only `RUN_MODE`, `MAX_EPOCHS` and `BATCH_SIZE` normally need touching.
There is no step budget — every epoch ends with a full val, a checkpoint and
a push, so you can stop whenever `metrics_val_steps.csv` flattens.

**`RUN_MODE`** — `"probe"` runs the 3k-step A/B from the README (~3 h) so you can
compare fresh vs warm-start before committing 27 h. `"full"` runs the real budget.

In [ ]:
# ── what to run ───────────────────────────────────────────────────────────────
RUN_MODE      = "full"           # "full" = the real 50k run.
                                 # "probe" = 3k steps (~3 h) if you ever
                                 # want a cheap check before committing.
INIT_FROM     = "VivekKanjarla/fr5-pi0-delta"
                                 # HF model repo (best.pt is fetched) or a local
                                 # .pt path. None = fresh from pi0_base.
                                 # Warm-starting Sanding off the FR5 pick-place run:
                                 # same arm, same two cameras, same 7-dim state and
                                 # action, same delta_joint@5 recipe, and the same
                                 # chunk/stride — verified field by field against
                                 # that checkpoint's stored config. pi0_base has
                                 # never seen this rig; that checkpoint has spent 6
                                 # epochs on it, which is the expensive part to
                                 # relearn from 5.5 h of new data.
                                 # It is the epoch-6 / val_l1 0.3616 checkpoint, the
                                 # MINIMUM of that run's curve — not the epoch-23
                                 # end state, which had drifted to 0.755.
                                 # init_from loads weights ONLY: action_mean/std stay
                                 # this run's, and the optimizer is not restored.

# ── the fix ───────────────────────────────────────────────────────────────────
DELTA         = True             # actions as offsets from state  (openpi: 6,-1)
ACTION_STRIDE = 5                # chunk samples every Kth frame  -> 5x the signal
FRAME_STRIDE  = 15               # chunk STARTS every Kth frame. Raised 5->15: with
                                 # ACTION_STRIDE=5 a chunk spans 8.3 s, so starts 5
                                 # frames apart overlap ~98%. Cuts the epoch 3x.

# ── data ──────────────────────────────────────────────────────────────────────
# Two ways in. RAW_SOURCE is the opt-in raw-recorder path (same shape as
# LEROBOT_SOURCE in train_xr1_runpod.ipynb): a repo of episode_*/ folders holding
# data.csv + the mp4s, which the dataset cell runs through
# common/convert_episodes.py into a LeRobot tree at DATA_ROOT. Leave it "" when
# HF_DATASET_REPO already IS a LeRobot dataset.
RAW_SOURCE      = "VivekKanjarla/Sanding"
HF_DATASET_REPO = "Slifold/fr5-pick-place-lerobot-v2"   # only when RAW_SOURCE == ""
DATA_ROOT     = "/workspace/dataset"
RAW_ROOT      = "/workspace/dataset_raw"

# ── RAW_SOURCE knobs (ignored when RAW_SOURCE == "") ─────────────────────────
RAW_GLOB      = "episode_*"      # top level only. The Sponge/ subtree (13 episodes
                                 # with a different gripper payload but the same
                                 # instruction string) was removed upstream on
                                 # 2026-09-06; this glob would exclude it anyway.
                                 # "**/episode_*" folds back in anything nested.
RAW_EXCLUDE   = ()               # nothing to drop. The 2026-09-06 re-upload filled
                                 # in all 19 episodes that had been missing videos,
                                 # including 096 and 205 — verified: their mp4 frame
                                 # counts now match meta.json and the _ts.npy arrays
                                 # exactly. All 265 episodes carry the identical
                                 # complete file set, so all 265 convert.
                                 # (convert_episodes still guards both failure modes:
                                 # it skips an episode with no timestamp file, and
                                 # keeps episode_index dense when it does.)
MAX_SYNC_GAP  = 0.10             # seconds; drop frames with no control sample that
                                 # close. The recorder's gripper call BLOCKS the
                                 # control loop ~1.85 s while the cameras keep
                                 # rolling at 30 Hz, so ~50 frames collapse onto one
                                 # frozen row — ~20% of every episode, always at the
                                 # grasp. Train on it and the policy learns to stop
                                 # dead when a grasp is imminent; a stopped policy
                                 # sees an unchanging scene and never restarts.
                                 # Cost: rows either side of a cut are ~1.8 s apart,
                                 # so the ~6 chunks per episode that straddle one
                                 # span more wall-clock than CHUNK_SIZE implies.
                                 # None = keep everything (old behaviour).
CHUNK_SIZE    = 10               # openpi's fine-tune configs use 10-15, NOT the
                                 # class default 50 (pi0_libero action_horizon=10,
                                 # pi05_libero 15, DROID 16). Measured on this data:
                                 # the normaliser is pooled over all chunk positions,
                                 # so chunk 50 x stride 5 = 8.3 s inflated it to
                                 # 9.23 deg and dropped SNR at entry 0 (the action
                                 # actually executed) to 0.0124. chunk 10 x stride 5
                                 # = 1.67 s -> std 2.59, SNR 0.0444, 47x absolute.
CAMERAS       = ["wrist_cam", "scene_cam"]
# photometric = brightness 0.3 + contrast 0.3 + mild blur. NOTHING that moves
# pixels or changes colour, and that is deliberate:
#   * "crops" always jitters saturation x[0.5,1.5] and hue +-28.8 deg, and
#     greyscales 5% of samples outright. On the FR5 pick-place set that was fatal
#     because the task was literally colour ("put the BLUE blocks in the BROWN
#     tray") — measured saturation 0.000-0.268 against a 0.084 baseline. Sanding
#     names no colour, but a bare metal object still has to be picked out from
#     black sandpaper on brown MDF, so the hue and greyscale draws stay off.
#   * both cameras are FIXED (wrist D405 + fixed scene D435i), so RandomCrop
#     simulates camera movement that never happens and shifts where objects sit in
#     pixel space, breaking the "object at pixel (x,y) -> reach there" mapping.
#     See the comment on the photometric branch in common/dataset.py.
AUG_LEVEL     = "photometric"
AUG_PROB      = 0.25              # fraction of samples that get jittered at all;
                                 # the rest pass through clean. openpi uses 1.0
                                 # (their ColorJitter is gated on nothing), but the
                                 # lab lighting varies far less than +-30%, so
                                 # augmenting every sample spends capacity on
                                 # variation that never happens. 0.0 = none.
VAL_FRAC      = 0.05
TASK_TEXT     = "Place the metal object on the sandpaper."   # must match the
                                 # dataset string exactly — asserted below

# ── model / finetune ──────────────────────────────────────────────────────────
POLICY        = "pi0"
PRETRAINED    = "lerobot/pi0_base"

# openpi's PRIMARY recipe is full fine-tune. Their README lists
#   Fine-Tuning (Full) > 70 GB  A100 (80GB)/H100     <- this pod
#   Fine-Tuning (LoRA) > 22.5 GB  RTX 4090
# and every LoRA config they ship is named *_low_mem_finetune. On an A100-80 there
# is no reason to run the 4090 recipe, so "full" is the default here.
#
#   full         everything trains (openpi's recipe). LORA_RANK ignored.
#   lora         adapters on Gemma; SigLIP + expert stay FULLY trainable, which is
#                openpi's actual shape (get_freeze_filter only freezes ".*llm.*",
#                and the model is nnx.Dict(llm=..., img=...) so img is never frozen)
#   expert_only  the 24 GB fallback. NOT an openpi recipe — no openpi SFT config
#                freezes the whole VLM. Use only if nothing else fits.
FINETUNE_MODE = "full"           # full | lora | expert_only
LORA_RANK     = 16               # used only when FINETUNE_MODE == "lora"
LORA_ALPHA    = 32
PROPRIO_MODE  = "full"           # full | dropout | none
COMPILE       = True
COMPILE_MODE  = "default"        # 'max-autotune' is lerobot's default and is the
                                 # one that had to be backed out on sm_100; try it
                                 # on A100 only after a default-mode run is green.

# ── optimisation ──────────────────────────────────────────────────────────────
BATCH_SIZE    = 32               # 48-64 should fit on A100-80 WITH grad ckpt on.
                                 # LR is sqrt-scaled automatically below.
LR            = None             # None -> scaled_lr(BATCH_SIZE) off openpi's 2.5e-5@32
WARMUP_STEPS  = 1000
LR_FLOOR      = 0.1              # cosine lands here x peak, not at 0
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
GRAD_CKPT     = True             # required: no-ckpt OOMs on 80 GB even at batch 24
SEED          = 42

# ── dataloader (common/train.py hardcodes num_workers=2 — that is the bottleneck)
NUM_WORKERS   = 8
PREFETCH      = 4

# ── logging / output ──────────────────────────────────────────────────────────
CKPT_DIR      = Path("/workspace/checkpoints_pi0_delta")
MODEL_REPO    = "auto"
USE_WANDB     = True
# Live logging needs a key. Jupyter inherits the environment when the KERNEL
# starts, so `export WANDB_API_KEY=...` in the pod shell after that is invisible
# here — either export it before launching Jupyter, or just paste it below.
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "") or ""   # <- or paste it here
LOG_EVERY     = 25               # steps between metrics_steps.csv rows (windowed mean)
VAL_EVERY     = 500              # sub-val, in optimizer steps
SAVE_EVERY    = 2000

# Epoch-based. There is no step budget: every epoch ends with a full val, a
# checkpoint, and a push, so you can watch metrics_val_steps.csv and stop whenever
# the curve flattens. MAX_EPOCHS only sets the cosine horizon and the hard stop.
# For scale: openpi's reference budget is ~8 epochs; ~45 min/epoch at batch 32.
MAX_EPOCHS    = 10               # was 20. The 2026-07-31 pick-place run bottomed at
                                 # epoch 6 (val 0.3616) and then drifted to 0.755 by
                                 # epoch 23 — 109% worse, and ~74% of the pod hours
                                 # bought nothing. Sanding is LESS diverse than that
                                 # set (one instruction, one object, one target), and
                                 # warm-starting begins nearer a fitted solution, so
                                 # it should turn sooner, not later.
EARLY_STOP_PATIENCE = 3          # stop after this many epochs with no full-val
                                 # improvement. 0 disables. best.pt already tracks
                                 # the minimum, so this only stops burning GPU past
                                 # the point the curve turned.
if RUN_MODE == "probe":
    MAX_EPOCHS, WARMUP_STEPS = 1, min(WARMUP_STEPS, 200)

from speedups import scaled_lr, FULL_LORA_TARGETS, finetune_flags
LR = LR or scaled_lr(BATCH_SIZE)
FT = finetune_flags(FINETUNE_MODE, LORA_RANK)
print(f"finetune_mode={FINETUNE_MODE}  ->  {FT}")
print(f"RUN_MODE={RUN_MODE}  max_epochs={MAX_EPOCHS}  batch={BATCH_SIZE}  "
      f"lr={LR:.2e}  warmup={WARMUP_STEPS}")
print(f"delta={DELTA}  action_stride={ACTION_STRIDE}  frame_stride={FRAME_STRIDE}")
print(f"horizon: chunk {CHUNK_SIZE} x stride {ACTION_STRIDE} = "
      f"{CHUNK_SIZE*ACTION_STRIDE/30:.1f} s predicted | execute 10 entries = "
      f"{10*ACTION_STRIDE/30:.2f} s open-loop before re-planning")

# Decide wandb NOW, not 20 minutes into training. Never interactive either way:
# with a key -> online; without -> offline to disk (sync afterwards). It will not
# sit at wandb's "(1) Create an account / (2) Use existing / (3) Don't visualize"
# prompt, which blocks on stdin and silently burns pod hours.
if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY          # "" is fine, means offline
    if WANDB_API_KEY:
        print(f"WANDB: ONLINE  (key ...{WANDB_API_KEY[-4:]})")
    else:
        print("WANDB: OFFLINE — no WANDB_API_KEY in this kernel's environment.")
        print("  to go live:  set WANDB_API_KEY above (or export it before starting")
        print("               Jupyter) and re-run THIS cell before training")
        print("  or after:    wandb sync <CKPT_DIR>/wandb/offline-run-*")
        print("  either way nothing is lost — the CSVs are the source of truth")
else:
    print("WANDB: disabled (USE_WANDB=False)")
print(f"init_from={INIT_FROM}")

## 2 · Dataset

In [ ]:
import json
from pathlib import Path
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq

# ALWAYS sync — do not skip when the tree exists. snapshot_download is incremental
# (unchanged videos are not re-fetched), and a pod with a persistent volume would
# otherwise keep a stale copy: the FR5 instruction vocabulary changed on 2026-07-30
# (400 unique per-episode strings -> 9 shared sentences), so a cached tree from
# before that silently trains on the old, episode-identifying labels.
if RAW_SOURCE:
    # Raw recorder format -> LeRobot. The depth .npz was deleted upstream on
    # 2026-09-06 (72 GB -> 28 GB), so ignore_patterns is now a no-op here — kept
    # because nothing downstream reads depth and a re-upload should not pull it.
    snapshot_download(RAW_SOURCE, repo_type="dataset", local_dir=RAW_ROOT,
                      ignore_patterns=["*_depth.npz"], max_workers=16)
    import convert_episodes
    convert_episodes.convert(
        Path(RAW_ROOT), Path(DATA_ROOT), extract_frames=False, max_frames=None,
        action_space="joint",        # DELTA is applied by DeltaJointDataset, not here
        glob_pattern=RAW_GLOB, exclude=RAW_EXCLUDE, max_sync_gap=MAX_SYNC_GAP,
        cameras=tuple(f"observation.images.{c}" for c in CAMERAS))
else:
    snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=DATA_ROOT)

_tasks = pq.read_table(Path(DATA_ROOT) / "meta" / "tasks.parquet").to_pandas()
_info  = json.loads((Path(DATA_ROOT) / "meta" / "info.json").read_text())
_n_ep  = _info["total_episodes"]
print(f"dataset at {DATA_ROOT}: {_n_ep} episodes, {_info['total_frames']:,} frames, "
      f"{len(_tasks)} instructions ({_n_ep / len(_tasks):.1f} episodes each)")
for _s in _tasks.task:
    print(f"    {_s}")

# The instructions must be SHARED. One string per episode means the instruction
# identifies the episode instead of the task — the thing the 2026-07-30 rewrite
# fixed. Tripping this almost always means a stale cached DATA_ROOT.
assert len(_tasks) * 2 <= _n_ep, (
    f"{len(_tasks)} instructions for {_n_ep} episodes — not shared. Delete "
    f"{DATA_ROOT} and re-run this cell to pull the current revision.")

# episode_index must be dense 0..n-1. FR5Dataset.episode_split hands out
# range(total_episodes), so a hole silently drops real episodes from BOTH splits —
# which is exactly what happened while convert_episodes numbered from
# enumerate(ep_dirs) and Sanding skipped 19 videoless episodes. Verify, don't trust.
_eps = pq.read_table(
    Path(DATA_ROOT) / "meta/episodes/chunk-000/file-000.parquet").to_pandas()
assert sorted(_eps.episode_index.tolist()) == list(range(_n_ep)), (
    f"episode_index is not dense 0..{_n_ep - 1} — episode_split would drop episodes")

# A single instruction is legal (Sanding is one task), but then the language input
# is a constant: this is a single-task policy and deploy must prompt with the exact
# string it trained on.
if len(_tasks) == 1:
    assert TASK_TEXT == _tasks.task.iloc[0], (
        f"TASK_TEXT {TASK_TEXT!r} != the dataset's only instruction "
        f"{_tasks.task.iloc[0]!r} — deploy would prompt with a string never seen")
    print(f"\n  single-task dataset: language is constant, TASK_TEXT matches")


In [ ]:
from dataset import FR5Dataset
from dataset_delta import DeltaJointDataset

n_eps = int(FR5Dataset(DATA_ROOT, use_image=False).info["total_episodes"])
train_eps, val_eps = FR5Dataset.episode_split(n_eps, VAL_FRAC, SEED)

def build(eps, aug):
    # pad_resize=True MUST match CFG["model"]["pad_resize"] below — deploy reads
    # that flag to choose its own transform, and squashing here while deploy
    # letterboxes is the train/inference mismatch that invalidated every
    # pre-2026-07-28 robot trial.
    return DeltaJointDataset(DATA_ROOT, CHUNK_SIZE, True, (224, 224),
                             episode_indices=eps, aug_level=aug,
                             frame_stride=FRAME_STRIDE, pad_resize=True,
                             aug_prob=AUG_PROB,
                             action_stride=ACTION_STRIDE, delta=DELTA)

train_ds, val_ds = build(train_eps, AUG_LEVEL), build(val_eps, "none")
IMAGE_KEYS = [f"observation.images.{c}" for c in CAMERAS]
stats = train_ds.get_stats()

print(f"episodes  train {len(train_eps)}  val {len(val_eps)}")
print(f"samples   train {len(train_ds)}  val {len(val_ds)}")
print(f"action_space recorded: {train_ds.info['action_space']!r}")
print(f"action_std (joints):   {stats['action_std'][:6].round(3)}")
print(f"chunk spans {CHUNK_SIZE * ACTION_STRIDE / 30:.1f} s of robot time")

### Sanity: is the normaliser actually tighter?

If `action_std` here is not clearly below the absolute run's ~13°, the whole point
of this notebook has been lost — stop and find out why before burning GPU hours.

In [ ]:
import numpy as np
abs_std = FR5Dataset(DATA_ROOT, CHUNK_SIZE, False, episode_indices=train_eps,
                     frame_stride=FRAME_STRIDE).get_stats()["action_std"][:6].mean()
new_std = stats["action_std"][:6].mean()
print(f"absolute action_std {abs_std:6.3f} deg")
print(f"this run  action_std {new_std:6.3f} deg   -> {abs_std / new_std:.1f}x tighter")
assert new_std < abs_std, "normaliser did NOT tighten — check DELTA/ACTION_STRIDE"

### Look at what the model will actually see

Images are **episode starts**: the pod has no extracted frames, so every load is
a `cv2.VideoCapture` seek into a network-volume mp4, and a random sample seeks
thousands of frames deep. The instruction counts and the `|delta|`-by-position
profile below are read straight from the parquet — vectorised, zero image decodes.


In [ ]:
import time as _time
import matplotlib.pyplot as plt
import numpy as np, torch, pyarrow.parquet as pq

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
show = lambda t: (t.cpu() * _STD + _MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

# The pod has NO extracted frames (the Hub ships videos only), so every image load
# is cv2.VideoCapture on a network-volume mp4 + a seek. A random sample lands up
# to ~5,000 frames into an episode and the seek crawls — that is what made this
# cell take forever. Episode-START samples seek to frame ~0, which is free.
view_ds = build(train_eps, "none")            # no-aug: show the DATA, not the aug
rng = np.random.default_rng(0)
_ep_first = {}                                 # episode -> first sample index
for _i, (_ep, _fr, _lo, _hi) in enumerate(view_ds._samples):
    if _ep not in _ep_first:
        _ep_first[_ep] = _i
picks = [_ep_first[e] for e in rng.choice(sorted(_ep_first), size=4, replace=False)]

fig, axes = plt.subplots(4, len(CAMERAS), figsize=(4.0 * len(CAMERAS), 3.2 * 4),
                         squeeze=False)
for r, idx in enumerate(picks):
    _t0 = _time.perf_counter()
    smp = view_ds[int(idx)]
    load_s = _time.perf_counter() - _t0
    for c, cam in enumerate(CAMERAS):
        key, ax = f"observation.images.{cam}", axes[r][c]
        if key in smp:
            im = show(smp[key])
            ax.imshow(im)
            bars = int((im.sum(2) == 0).all(1).sum())
            sat = float((im.max(2) - im.min(2)).mean())
            ax.set_title(f"{cam}  {bars}px bars  colour={sat:.3f}  t={load_s:.1f}s",
                         fontsize=9)
        ax.axis("off")
    st, a0, aN = (smp["observation.state"].numpy(),
                  smp["action"][0].numpy(), smp["action"][-1].numpy())
    print(f"sample {int(idx):6d} (episode start)  {smp['task']!r}")
    print(f"    state J1-J6 {np.round(st[:6], 2)}  grip {st[6]:.2f}")
    print(f"    delta[0]    {np.round(a0[:6], 3)}  grip {a0[6]:.2f}   <- now")
    print(f"    delta[{CHUNK_SIZE-1}]    {np.round(aN[:6], 3)}  grip {aN[6]:.2f}   "
          f"<- {CHUNK_SIZE*ACTION_STRIDE/30:.1f}s ahead")
plt.tight_layout(); plt.show()
print("images are EPISODE STARTS — chosen so the mp4 seek is free on a pod with "
      "no extracted frames; t= in each title is that row's actual load time")
print(f"aug_level={AUG_LEVEL!r} aug_prob={AUG_PROB} -> {AUG_PROB:.0%} of TRAINING "
      f"samples jittered (display above is un-augmented)")

# ── stats: fully vectorised, zero image decodes, zero __getitem__ loops ───────
_df = view_ds.df
_tasks = pq.read_table(f"{DATA_ROOT}/meta/tasks.parquet").to_pandas()
_i2t = dict(zip(_tasks.task_index, _tasks.task))
_tr_mask = _df.episode_index.isin(train_eps)
print(f"\ninstruction distribution over all {int(_tr_mask.sum()):,} train frames:")
for ti, n in _df.loc[_tr_mask, "task_index"].value_counts().sort_index().items():
    print(f"    {n:7,d} frames  {_i2t[ti]}")

_act = np.stack(_df["action"].to_numpy())[:, :6]
_sta = np.stack(_df["observation.state"].to_numpy())[:, :6]
_starts = np.array([t for _, t, _, _ in view_ds._samples])
_ends = np.array([e for _, _, _, e in view_ds._samples])
_k = np.arange(CHUNK_SIZE) * ACTION_STRIDE
_rows = np.minimum(_starts[:, None] + _k[None, :], _ends[:, None] - 1)
_delta = np.abs(_act[_rows] - _sta[_starts][:, None, :]).mean(axis=(0, 2))
print(f"\n|delta| by chunk position (deg, all {len(_starts):,} samples):")
for k in sorted({0, 1, CHUNK_SIZE // 4, CHUNK_SIZE // 2, CHUNK_SIZE - 1}):
    print(f"    entry {k:2d} ({k*ACTION_STRIDE/30:5.2f}s ahead): {_delta[k]:.3f}")
print("    entry 0 tiny and the last large = the delta transform is working")


## 3 · DataLoader — measure before you tune

`common/train.py` hardcodes `num_workers=2`. The v2 notebook measured **9.2 s/step
starved vs ~4.7 s/step compute-bound** — half the GPU's life spent waiting on JPEGs.

Compare `s/batch` below against your `s/step` in section 6. Large fraction → raise
`NUM_WORKERS`. **Do not raise `BATCH_SIZE` while you are input-bound**; it will not
help and it changes the LR, which confounds the A/B.

In [ ]:
from torch.utils.data import DataLoader
from speedups import loader_kwargs, probe_input_bound, shm_size_mb

# loader_kwargs sets torch.multiprocessing "file_system" sharing. RunPod caps
# /dev/shm at 64 MB by default, and 8 workers x prefetch 4 x batch 32 x 2 cameras
# is ~1.2 GB of in-flight tensors -> every worker dies at once, usually at the
# second epoch:  RuntimeError: DataLoader worker (pid(s) ...) exited unexpectedly
# If workers still die, set NUM_WORKERS = 0 (slower, but cannot fail this way).
_shm = shm_size_mb()
print(f"/dev/shm: {_shm:.0f} MB" if _shm else "/dev/shm: n/a")
if _shm and _shm < 1000:
    print("  (small — file_system sharing is what makes NUM_WORKERS>0 safe here)")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          **loader_kwargs(NUM_WORKERS, PREFETCH))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          **loader_kwargs(max(2, NUM_WORKERS // 2), PREFETCH))

_ = probe_input_bound(train_loader, n=20)

## 4 · Model

In [ ]:
import importlib.util
from speedups import enable_compile, init_from

spec = importlib.util.spec_from_file_location(
    f"policy_{POLICY}", REPO_DIR / "policies" / POLICY / "model.py")
policy_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(policy_mod)

if COMPILE:
    enable_compile(policy_mod, COMPILE_MODE)
    # LoRA is injected AFTER PI0Policy.__init__ compiles forward/sample_actions,
    # so the first few steps recompile once as the guards fail. Expect a slow
    # start (~1-3 min); judge s/step only after step ~50.
    print(f"[compile] on, mode={COMPILE_MODE} (first steps recompile — ignore s/step early)")

CFG = {
    "dataset": {"root": DATA_ROOT, "chunk_size": CHUNK_SIZE, "use_image": True,
                "image_size": [224, 224], "camera_names": CAMERAS,
                "val_frac": VAL_FRAC, "aug_level": AUG_LEVEL,
                "frame_stride": FRAME_STRIDE},
    "model": {
        "state_dim": 7, "action_dim": 7, "pretrained": PRETRAINED,
        **FT,                                    # rank / freeze flags for the mode
        "vlm_lora_alpha": LORA_ALPHA, "vlm_lora_dropout": 0.05,
        # BOTH towers. The old list used Gemma names only, so SigLIP got q/k/v and
        # nothing else -> exactly the 207 pairs in the deploy gate log.
        "vlm_lora_targets": list(FULL_LORA_TARGETS),
        "paligemma_variant": "gemma_2b", "action_expert_variant": "gemma_300m",
        "max_state_dim": 32, "max_action_dim": 32, "tokenizer_max_length": 48,
        "num_inference_steps": 10,
        # Inference-only receding horizon: execute 10 of the 50 chunk entries, then
        # re-plan. With ACTION_STRIDE=5 each entry is held 5 control steps, so 10
        # entries = 50 control steps = 1.67 s open-loop. None would execute the whole
        # chunk = 250 control steps = 8.3 s blind, and the far entries are 8-second
        # extrapolations (measured |delta| 10.8 deg at entry 49) — that is the
        # mechanism behind the 2026-07-28 wandering. 1.67 s matches both the
        # pre-ACTION_STRIDE default and the --n-action-steps 10 that launcher used.
        # Overridable at deploy without retraining.
        "n_action_steps": 10,
        "dtype": "bfloat16", "gradient_checkpointing": GRAD_CKPT,
        "quantize": "none", "proprio_mode": PROPRIO_MODE,
        "proprio_dropout_rate": 0.3,
        "prompt_newline": True, "pad_resize": True,          # format v3
    },
    "training": {"batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY,
                 "grad_clip": GRAD_CLIP, "seed": SEED, "max_epochs": MAX_EPOCHS,
                 "warmup_steps": WARMUP_STEPS, "device": "cuda"},
}

# The checkpoint's pad_resize is what deploy trusts; make sure the data actually
# went through the letterbox. A mismatch here is silent and ruins the rollout.
assert CFG["model"]["pad_resize"] == train_ds.pad_resize, (
    f'CFG pad_resize={CFG["model"]["pad_resize"]} but the dataset used '
    f'{train_ds.pad_resize} — train/deploy image formats would disagree')

device = torch.device("cuda")
torch.manual_seed(SEED)
model = policy_mod.build_model(CFG, stats, device)

# LoRA mode only: replicate openpi's freeze shape. lerobot has no "freeze the LLM
# but not the vision tower" flag, so do it by name. NOT applied for full/expert_only.
if FINETUNE_MODE == "lora":
    from speedups import freeze_llm_keep_vision
    freeze_llm_keep_vision(model)

n_lora    = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
vis_train = sum(p.numel() for n, p in model.named_parameters()
                if "vision_tower" in n and p.requires_grad)
print(f"mode={FINETUNE_MODE}  LoRA {n_lora/1e6:.1f}M  trainable {trainable/1e6:.0f}M  "
      f"frozen {frozen/1e6:.0f}M")
print(f"vision tower trainable: {vis_train/1e6:.0f}M params")
# The camera rig is new to pi0_base, so a frozen vision tower is the single worst
# thing to freeze. openpi never does it — fail loudly rather than train blind.
assert vis_train > 0 or FINETUNE_MODE == "expert_only", \
    "vision tower is FROZEN — openpi's freeze filter never freezes it; check FINETUNE_MODE"

### Warm-start (optional)

`init_from` loads **weights only** and deliberately drops `action_mean` /
`action_std`. They are registered as *buffers*, so a plain `load_state_dict` would
restore the old **absolute** stats over the delta ones — targets collapse to 0.12
of a unit, the 5× gain is exactly cancelled, and the loss still goes down. Silent.

Optimizer state is never restored: Adam's moments are calibrated to the old scale.

In [ ]:
if INIT_FROM:
    src = INIT_FROM
    if not Path(src).exists():          # treat as an HF model repo -> best.pt
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(INIT_FROM, "best.pt", token=HF_TOKEN)
        print(f"fetched {INIT_FROM}/best.pt -> {src}")
    before_std = model.action_std.clone()
    # a real weight, to prove the load actually landed rather than no-op'ing
    probe_name, probe_before = next(
        (n, p.detach().clone()) for n, p in model.named_parameters() if p.numel() > 1e6)

    init_from(model, src, device=device)

    assert torch.allclose(model.action_std, before_std), "stats buffers leaked through!"
    after = dict(model.named_parameters())[probe_name]
    assert not torch.allclose(after, probe_before), (
        f"{probe_name} is unchanged after init_from — nothing was actually loaded")
    print("verified: weights loaded, and this run's Sanding stats survived")
else:
    print("fresh from", PRETRAINED)


## 5 · Optimizer + schedule

`common/train.py` builds a bare AdamW with **no scheduler at all** — the
warmup+cosine the 30k run used lived only in the old notebook. It is wired here.

On warm-start do **not** lower the peak LR: the action expert sits in a basin built
around "copy the state", and leaving it is the whole point. A reduced LR glues it
in place and you would wrongly conclude warm-starting does not work.

In [ ]:
from speedups import warmup_cosine

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
STEPS_PER_EPOCH = max(1, len(train_ds) // BATCH_SIZE)
TOTAL_STEPS = MAX_EPOCHS * STEPS_PER_EPOCH      # cosine horizon only, not a budget
scheduler = warmup_cosine(optimizer, TOTAL_STEPS, WARMUP_STEPS, LR_FLOOR)
print(f"AdamW lr={LR:.2e} wd={WEIGHT_DECAY} | warmup {WARMUP_STEPS} -> cosine to "
      f"{LR*LR_FLOOR:.2e} over {TOTAL_STEPS:,} steps "
      f"({STEPS_PER_EPOCH:,}/epoch x {MAX_EPOCHS} epochs)")
print("stopping early is fine — the LR simply will not have finished decaying")

## 6 · Train

Step-based, not epoch-based — with `FRAME_STRIDE` in play "epoch" is not a
comparable unit across runs. `val_l1` is the flow-matching MSE on **normalized**
actions.

**It will be higher than the old run's 0.0339, and that is correct.** The absolute
target contained a large trivially-predictable component (the current position);
removing it removes the easy win. Do not tune toward the old number — that number
was mostly measuring the shortcut.

In [ ]:
import csv, time, math

CKPT_DIR.mkdir(parents=True, exist_ok=True)
# wandb is ALWAYS enabled and can NEVER prompt. Without a key it runs offline
# (writes to CKPT_DIR/wandb, sync later with `wandb sync <dir>`); with one it goes
# live. WANDB_SILENT + the explicit mode are what stop `wandb.init` from opening
# its "(1) Create an account / (2) Use an existing / (3) Don't visualize" menu,
# which blocks on stdin and quietly burns pod hours.
run = None
if USE_WANDB:
    os.environ["WANDB_SILENT"] = "true"
    os.environ["WANDB_CONSOLE"] = "off"
    os.environ["WANDB_DIR"] = str(CKPT_DIR)
    # mode was resolved and announced in the params cell
    os.environ["WANDB_MODE"] = "online" if os.environ.get("WANDB_API_KEY") else "offline"
    import wandb
    try:
        run = wandb.init(project="fr5-vla-benchmark",
                         name=f"pi0-delta{int(DELTA)}-as{ACTION_STRIDE}-bs{BATCH_SIZE}-{RUN_MODE}",
                         config={**CFG["model"], **CFG["training"],
                                 "action_stride": ACTION_STRIDE, "delta": DELTA,
                                 "frame_stride": FRAME_STRIDE,
                                 "train_samples": len(train_ds),
                                 "val_samples": len(val_ds),
                                 "init_from": str(INIT_FROM)},
                         settings=wandb.Settings(init_timeout=60, silent=True))
        print(f"wandb: {os.environ['WANDB_MODE']}  ->  {run.url if WANDB_API_KEY else CKPT_DIR/'wandb'}")
    except Exception as e:
        # logging must never take the run down; the CSVs are the source of truth
        print(f"wandb unavailable ({type(e).__name__}: {e}) — continuing, CSVs still written")
        run = None

# Two CSVs, same schema as the pre-delta notebooks so old plots keep working:
#   metrics_steps.csv      every LOG_EVERY steps, WINDOWED means (not the noisy
#                          instantaneous value the stdout line shows)
#   metrics_val_steps.csv  one row per sub-val
STEPS_CSV = CKPT_DIR / "metrics_steps.csv"
VAL_CSV   = CKPT_DIR / "metrics_val_steps.csv"
if not STEPS_CSV.exists():
    STEPS_CSV.write_text("step,epoch,train_l1,grad_norm,lr,samples_per_s,s_per_step\n")
if not VAL_CSV.exists():
    VAL_CSV.write_text("step,epoch,val_l1_sub\n")

print(f"{len(train_ds):,} train samples / batch {BATCH_SIZE} = {STEPS_PER_EPOCH:,} "
      f"steps per epoch  |  {MAX_EPOCHS} epochs max = {TOTAL_STEPS:,} steps")
print(f"logging: metrics_steps.csv every {LOG_EVERY} steps · sub-val every "
      f"{VAL_EVERY} · full val + checkpoint + push every epoch")

# ── per-epoch push ────────────────────────────────────────────────────────────
# best.pt is ~6.9 GB and uploads at ~4 MB/s (~29 min) against a ~45 min epoch, so
# the checkpoint goes up in the BACKGROUND (run_as_future) or it would add ~64%
# to wall-clock. The CSVs are kilobytes and go up synchronously every epoch, so
# the loss curves are on the Hub even if a checkpoint push is still in flight.
from huggingface_hub import HfApi
_api = HfApi()
PUSH_REPO = MODEL_REPO if MODEL_REPO != "auto" else f"{HF_USER}/fr5-pi0-delta"
_api.create_repo(PUSH_REPO, private=True, exist_ok=True, repo_type="model")
print(f"pushing every  epoch -> https://huggingface.co/{PUSH_REPO}")
_ckpt_future = None


def push_epoch(ep, val_l1, is_best):
    """CSVs synchronously; best.pt in the background when it improved."""
    global _ckpt_future
    for f in ("metrics_steps.csv", "metrics_val_steps.csv"):
        if (CKPT_DIR / f).exists():
            try:
                _api.upload_file(path_or_fileobj=str(CKPT_DIR / f), path_in_repo=f,
                                 repo_id=PUSH_REPO,
                                 commit_message=f"metrics @ epoch {ep}")
            except Exception as e:
                print(f"  [push] {f} failed ({type(e).__name__}) — local copy kept")
    if not is_best:
        return
    if _ckpt_future is not None and not _ckpt_future.done():
        print("  [push] previous best.pt still uploading — skipping this one")
        return
    try:
        _ckpt_future = _api.upload_file(
            path_or_fileobj=str(CKPT_DIR / "best.pt"), path_in_repo="best.pt",
            repo_id=PUSH_REPO, run_as_future=True,
            commit_message=f"best.pt epoch {ep} val_l1 {val_l1:.4f}")
        print(f"  [push] best.pt uploading in background (epoch {ep}, val {val_l1:.4f})")
    except Exception as e:
        print(f"  [push] best.pt failed ({type(e).__name__}) — local copy kept")


def save(name, ep, val_l1):
    torch.save({"model_state": model.state_dict(), "config": CFG, "stats": stats,
                "policy": POLICY, "epoch": ep, "step": step, "val_l1": val_l1,
                "action_space": train_ds.info["action_space"]}, CKPT_DIR / name)


@torch.no_grad()
def sub_val(n_batches=20):
    """Quick intra-epoch val: first n_batches of the val split."""
    model.eval(); tot = k = 0
    for b in val_loader:
        img = {kk: b[kk].to(device, non_blocking=True) for kk in IMAGE_KEYS if kk in b}
        _, l1, _ = model(b["observation.state"].to(device), b["action"].to(device),
                         b["action_is_pad"].to(device), img, task=list(b["task"]))
        tot += l1; k += 1
        if k >= n_batches:
            break
    model.train()
    return tot / max(k, 1)


@torch.no_grad()
def full_val():
    """Every val batch, not the 20-batch sub-val."""
    model.eval(); tot = k = 0
    for b in val_loader:
        img = {kk: b[kk].to(device, non_blocking=True) for kk in IMAGE_KEYS if kk in b}
        _, l1, _ = model(b["observation.state"].to(device), b["action"].to(device),
                         b["action_is_pad"].to(device), img, task=list(b["task"]))
        tot += l1; k += 1
    model.train()
    return tot / max(k, 1)


# ── train ─────────────────────────────────────────────────────────────────────
model.train()
step, best, stop = 0, float("inf"), False
bad_epochs = 0          # consecutive epochs with no full-val improvement
t0, n_win, loss_win, gn_win = time.perf_counter(), 0, 0.0, 0.0
t_epoch = time.perf_counter()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        for batch in train_loader:
            obs = batch["observation.state"].to(device, non_blocking=True)
            act = batch["action"].to(device, non_blocking=True)
            pad = batch["action_is_pad"].to(device, non_blocking=True)
            img = {k: batch[k].to(device, non_blocking=True)
                   for k in IMAGE_KEYS if k in batch}

            loss, l1, _ = model(obs, act, pad, img, task=list(batch["task"]))
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
            step += 1
            n_win += 1; loss_win += l1; gn_win += float(gn)

            if step % LOG_EVERY == 0:
                sps = (time.perf_counter() - t0) / max(n_win, 1)
                l1_avg, gn_avg = loss_win / n_win, gn_win / n_win
                lr_now = optimizer.param_groups[0]["lr"]
                epoch_f = step / STEPS_PER_EPOCH
                eta = (TOTAL_STEPS - step) * sps / 3600
                print(f"step {step:6d}  ep {epoch_f:5.2f}/{MAX_EPOCHS}  "
                      f"l1={l1_avg:.4f}  lr={lr_now:.2e}  gn={gn_avg:.2f}  "
                      f"{sps:.2f}s/step  eta {eta:.1f}h")
                with open(STEPS_CSV, "a") as f:
                    csv.writer(f).writerow([step, f"{epoch_f:.4f}", f"{l1_avg:.6f}",
                                            f"{gn_avg:.4f}", f"{lr_now:.3e}",
                                            f"{BATCH_SIZE / sps:.2f}", f"{sps:.3f}"])
                if run: run.log({"train_l1": l1_avg, "lr": lr_now,
                                 "grad_norm": gn_avg, "s_per_step": sps,
                                 "epoch": epoch_f}, step=step)
                t0, n_win, loss_win, gn_win = time.perf_counter(), 0, 0.0, 0.0

            if step % VAL_EVERY == 0:
                v = sub_val()
                print(f"          sub-val l1={v:.4f}")
                with open(VAL_CSV, "a") as f:
                    csv.writer(f).writerow([step, f"{step/STEPS_PER_EPOCH:.4f}",
                                            f"{v:.6f}"])
                if run: run.log({"val_l1_sub": v}, step=step)
                t0, n_win, loss_win, gn_win = time.perf_counter(), 0, 0.0, 0.0

        # ── end of epoch: full val, checkpoint, push ──────────────────────────
        v = full_val()
        mins = (time.perf_counter() - t_epoch) / 60
        is_best = v < best
        if is_best:
            best = v
        print(f"\n=== epoch {epoch}/{MAX_EPOCHS}  step {step}  full-val l1={v:.4f}"
              f"{'  <- BEST' if is_best else f'  (best {best:.4f})'}  [{mins:.1f} min] ===")
        with open(VAL_CSV, "a") as f:
            csv.writer(f).writerow([step, f"{epoch}", f"{v:.6f}"])
        if run: run.log({"val_l1_full": v, "epoch_done": epoch}, step=step)
        save("last.pt", epoch, v)
        if is_best:
            save("best.pt", epoch, v)
        push_epoch(epoch, v, is_best)

        # Early stop. best.pt already holds the minimum, so training past the turn
        # only spends pod hours: the 2026-07-31 run kept going for 17 epochs after
        # its epoch-6 best while val drifted 109% worse.
        bad_epochs = 0 if is_best else bad_epochs + 1
        if EARLY_STOP_PATIENCE and bad_epochs >= EARLY_STOP_PATIENCE:
            stop = True
            print(f"\nearly stop: {bad_epochs} epochs with no improvement on "
                  f"{best:.4f} — best.pt holds it, stopping rather than burning GPU")
            break

        t_epoch = t0 = time.perf_counter()
        n_win, loss_win, gn_win = 0, 0.0, 0.0

except KeyboardInterrupt:
    stop = True
    print("\ninterrupted — saving and pushing before exit")
    v = full_val()
    save("last.pt", -1, v)
    if v < best:
        best = v; save("best.pt", -1, v)
    push_epoch("interrupt", v, v <= best)

if _ckpt_future is not None and not _ckpt_future.done():
    print("waiting for the background best.pt upload ...")
    _ckpt_future.result()
print(f"\n{'stopped' if stop else 'finished'} at step {step} "
      f"({step/STEPS_PER_EPOCH:.2f} epochs), best full-val l1 {best:.4f}")
print(f"https://huggingface.co/{PUSH_REPO}")


## 7 · Push

`create_repo` first — `upload_file` does **not** create repos, and HF returns 404
for both "missing" and "your token cannot see it".

`PUSH_RESUME=False` by default: `last.pt` carries optimizer state (~2× the size)
and you should not reuse those moments across an action-space change anyway.

In [ ]:
from huggingface_hub import HfApi

PUSH_RESUME = False
# SAME repo the training loop pushed to every epoch — one run, one repo. The old
# line computed fr5-pi0-delta-{RUN_MODE} here vs fr5-pi0-delta in the loop, so the
# final push landed in a different repo than the per-epoch checkpoints.
repo = PUSH_REPO
files = [f for f in ["best.pt", "metrics_steps.csv", "metrics_val_steps.csv"] + (["last.pt"] if PUSH_RESUME else [])
         if (CKPT_DIR / f).exists()]
gb = sum((CKPT_DIR / f).stat().st_size for f in files) / 1e9

card = f"""---
license: apache-2.0
tags: [robotics, vla, pi0, lerobot, fairino-fr5]
---
# {repo.split('/')[-1]}

- **action_space**: `{train_ds.info['action_space']}` — joint offsets from state,
  gripper absolute (openpi `make_bool_mask(6, -1)`); each chunk entry is
  {ACTION_STRIDE} frames apart and must be held {ACTION_STRIDE} control steps at deploy.
- **Deploy with** `python delta_joint/run.py deploy --checkpoint best.pt` — the plain
  `common/deploy.py` will NOT add the state back.
- Base `{PRETRAINED}` · LoRA r={LORA_RANK} on both towers · batch {BATCH_SIZE} ·
  lr {LR:.2e} (warmup {WARMUP_STEPS} + cosine) · {step} steps · best val_l1 {best:.4f}
- val_l1 is normalized-space MSE and is **not comparable** to absolute-action runs.
"""

print(f"pushing {files} (~{gb:.1f} GB) -> {repo}")
api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)      # <- the line the old one missed
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
api.upload_folder(folder_path=str(CKPT_DIR), repo_id=repo, allow_patterns=files)
print("done ->", f"https://huggingface.co/{repo}")

## 8 · Next

**If `RUN_MODE="probe"`**: sanity-check the `val_l1` curve in
`metrics_steps.csv` (decision 2026-07-30: fresh from `pi0_base` only — no
warm-start from the 30k absolute-action checkpoint; its action expert learned
exactly the shortcut this run removes). If the curve is sane, set
`RUN_MODE="full"` and rerun.

### Then, on the Linux GPU PC (robot side)

The same repo runs inference — clone it next to `so101-fr5-teleop`, same venv
recipe (`lerobot==0.5.1`). Everything below reads the recipe out of the
checkpoint (`delta_joint@5`), so it cannot be run at the wrong speed, without the
state added back, or with the int64-mask crash the training notebooks used to
patch by hand:

```bash
# 1. offline eval on held-out episodes (writes <ckpt_dir>/eval/ep*.npz)
python delta_joint/run.py eval --ckpt best.pt --episodes 4

# 2. the pre-registered pass/fail — exit 0 = book robot time, 1 = keep training
python delta_joint/gate.py <ckpt_dir>/eval

# 3. only if the gate passes: the robot
python delta_joint/run.py deploy --hf-repo <you>/fr5-pi0-delta-full --task "..."
```

Gate criteria (fixed in `delta_joint/gate.py` BEFORE the run, on purpose):
model beats the "don't move" baseline on every episode, and the predicted
gripper crosses 0.65 where the ground truth closes. The 30k run fails both
(1.656 vs 0.075 deg; gripper max 0.154). `val_l1` is deliberately not consulted.

Gate C from the old predeploy gate (instruction A/B) stays retired on this
dataset — see `delta_joint/README.md` §6.
